In [ ]:
# Multivariable Logistic Regression (ESBL)

This notebook explores predictors of ESBL status and checks model assumptions before fitting a multivariable logistic regression model.


# Notebook run instructions

This notebook should be run in the SDE after importing the cleaned `analysis_df`.

## Intended order

1. Load packages, project paths, and `analysis_df`.
2. Restrict to surgical infection episodes using `surgery_before_infection == 1`.
3. Keep required modelling columns, including:
   - `subject_id`
   - `sample_collection_date`
   - outcome: `esbl_status`
   - all candidate predictors.
4. Recode prophylaxis groups.
5. Remove unknown/missing categories as pre-specified.
6. Check cohort size, ESBL/non-ESBL counts, and repeated infection episodes per patient.
7. Encode categorical predictors using dummy variables with `prefix_sep="__"`.
8. Drop one reference category per categorical variable.
9. Check EPV, missingness, sparse categories, correlation, and VIF.
10. Fit main multivariable logistic regression on the full analysis cohort.
11. Report adjusted ORs, 95% CIs, p-values, and test-set AUC.
12. Plot ROC curve using test-set predictions only.
13. Run first-episode-only sensitivity analysis.
14. Compare main LR vs first-episode sensitivity ORs.
15. Run GEE clustered by `subject_id` as an additional sensitivity analysis.
16. Compare main LR vs GEE ORs.
17. Run exploratory stratified models by TFC and sample site only where there are sufficient ESBL and non-ESBL events.
18. Optionally produce calibration curve and confusion matrix if discussing prediction performance.
19. Export final tables:
   - main adjusted OR table
   - sensitivity comparison table
   - GEE comparison table
   - stratified model summaries
   - missingness/diagnostic outputs.

## Main interpretation rules

The primary result is the multivariable logistic regression adjusted OR table.

The test-set AUC describes discrimination/predictive performance, not causal association.

The full-data AUC should not be reported as the main AUC because it is optimistic.

Sensitivity analyses are used to check whether conclusions change when repeated patient episodes are handled differently.

Stratified analyses should be interpreted as exploratory, especially where event numbers are small.

P-values should not be interpreted alone; effect size, confidence intervals, clinical plausibility, and robustness across sensitivity analyses should also be considered.

## 1) Setup and Data Load


In [48]:
import numpy as np
import sys
from pathlib import Path
import pandas as pd
import matplotlib as plt 
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from patsy.builtins import Q 
import sklearn as sk
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve
from sklearn.model_selection import train_test_split
import matplotlib.dates as mdates
import matplotlib.ticker as mtick
import matplotlib.ticker as mticker

PROJECT_ROOT = Path.cwd().parent     
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from utils import data_cleaning_tools as dct
cfg_path="../configs/config.yaml"
cfg = dct.load_config(cfg_path)

df_path = PROJECT_ROOT / 'notebooks' / 'lr_data.csv'
analysis_df = pd.read_csv(df_path)

In [49]:
to_keep = ['age_at_admission',
 'colonisation',
 'copd',
 'days_from_admission_to_first_culture',
 'esbl_status',
 'ethnicity_desc',
 'hospital_exposure_90d',
 'organism_bug',
 'past_abx',
 'prophylaxis_group',
 'site',
 'surgery_length',
 'tfc',
 'type2_diabetes']

analysis_df= analysis_df[to_keep]

## Working Notes and TODO

- collinearity and multicollinearity (variables highly correalted) is bad bc model can't tell which variable is responsible for the changes in outcome 
    - coefficients become unstable + very larger CIs

- dummy variables - binary encodings of categorical variables
    - drop one category so you don't get a perfect linear relationship = one variable cannot be exactly predicted from others 

- what is EPV? (number of events/ number of variables)
     - needs to be >10


**Final intuitions**:

- Dummy variables → allow categories into regression
- Dropping one → avoids mathematical impossibility
- VIF → detects redundancy
- EPV → protects against overfitting
- AUROC → evaluates prediction, not causality

## 2) Feature Engineering and EPV Check

Create dummy variables for categorical predictors and calculate events-per-variable (EPV) to gauge model stability.


In [50]:
analysis_df.columns

In [51]:
# def get_epv(df, multi_lr_cols):


multi_lr_cols = cfg['univariable_lr']
continuous_vars = multi_lr_cols["continuous"]
categorical_vars = multi_lr_cols["categorical"]
outcome = multi_lr_cols["outcome"]

print(set(continuous_vars + categorical_vars) - set(analysis_df))

reference_categories = {
    "site": "blood",
    "organism_bug": "escherichia coli",
    "ethnicity_desc": "white",
    "tfc": "general_surgery",
    "prophylaxis_group": "cefuroxime + metronidazole",
}

X = analysis_df.drop(columns=["esbl_status"])
y = analysis_df["esbl_status"]

X_encoded = pd.get_dummies(
    X,
    columns=categorical_vars,
    drop_first=False,
    prefix_sep="__",
    dtype=int
)

for col, ref in reference_categories.items():
    X_encoded = X_encoded.drop(columns=f"{col}__{ref}", errors="ignore")

multi_lr_df = X_encoded.copy()
multi_lr_df["esbl_status"] = y.map({"ESBL": 1, "non-ESBL": 0}) if y.dtype == "object" else y
    
n_predictors = multi_lr_df.drop(columns=["esbl_status"]).shape[1]
epv = multi_lr_df["esbl_status"].sum() / n_predictors
print(f"Events per variable (EPV): {epv:.2f}")

## 3) VIF and Correlation Screen for Multicollinearity

Visualize the correlation matrix to identify highly correlated predictors before model fitting.

- use Pearson's corrrelation coefficient to screen for collinearity
- VIF - variance inflation factor 


In [52]:
# Define a threshold for the absolute value of correlation
threshold = 0.15

# Select predictors only (exclude outcome) using column-safe indexing
predictor_df = multi_lr_df.drop(columns=['esbl_status'], errors='ignore')

# Correlation among predictors
corr_matrix = predictor_df.corr().abs()

# Keep variables that have at least one correlation above threshold (excluding self-correlation)
relevant_vars = corr_matrix.columns[(corr_matrix > threshold).sum() > 1]

# Apply the threshold to the correlation matrix
filtered_corr = corr_matrix.loc[relevant_vars, relevant_vars]

plt.figure(figsize=(14, 10))
plt.imshow(filtered_corr, cmap='coolwarm', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(label='Correlation')
plt.xticks(range(len(filtered_corr.columns)), filtered_corr.columns, rotation=90)
plt.yticks(range(len(filtered_corr.index)), filtered_corr.index)
plt.tight_layout()
plt.show()

In [53]:
# Example predictor matrix--
X = multi_lr_df.copy()
X = X.drop(columns=['esbl_status'])

# Drop rows with NaN values
X = X.dropna()

# Add constant
X = sm.add_constant(X)

vif_df = pd.DataFrame()
vif_df['variable'] = X.columns
vif_df['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
vif_df = vif_df.sort_values(by="VIF", ascending=False)

print(vif_df[vif_df['VIF'] > 5])

## 5) Modeling and Diagnostics 

Fit the logistic regression model here and report: odds ratios, confidence intervals, goodness-of-fit, and discrimination metrics.


In [54]:
def run_multivariable_lr(df, col_dict, test_size = 0.2):
    import numpy as np
    import pandas as pd
    import statsmodels.api as sm
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import roc_auc_score, roc_curve

    outcome_col = col_dict['outcome']

    categorical_cols = col_dict['categorical']

    reference_categories = {
        "organism_bug": "escherichia coli",
        "site": "blood",
        "ethnicity_desc": "white",
        "prophylaxis_group": "cefuroxime + metronidazole",
        "tfc": "general_surgery"
    }

    X = df.drop(columns=[outcome_col])
    y = df[outcome_col]

    if y.dtype == "object":
        y = y.map({"ESBL": 1, "non-ESBL": 0})

    X = pd.get_dummies(
        X,
        columns=[c for c in categorical_cols if c in X.columns],
        prefix_sep="__",
        drop_first=False,
        dtype=int
    )

    for col, ref in reference_categories.items():
        X = X.drop(columns=f"{col}__{ref}", errors="ignore")

    X = X.apply(pd.to_numeric, errors="coerce")

    model_df = X.copy()
    model_df[outcome_col] = y.values
    model_df = model_df.dropna()

    y = model_df[outcome_col]
    X = model_df.drop(columns=[outcome_col])

    
    # remove 0 variance columns 
    
    X = X.loc[:, X.nunique() > 1]
    X = sm.add_constant(X, has_constant="add")

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=42,
        stratify=y #ESBL prevealance stays the same in train and test 
    )

    model = sm.Logit(y_train, X_train).fit(disp=False)

    y_pred = model.predict(X_test)
    auc = roc_auc_score(y_test, y_pred)

    # ROC curve coordinates 
    fpr, tpr, thresholds = roc_curve(y_test, y_pred)

    conf = model.conf_int()

    or_df = pd.DataFrame({
        "term": model.params.index,
        "OR": np.exp(model.params),
        "ci_lower": np.exp(conf[0]),
        "ci_upper": np.exp(conf[1]),
        "p_value": model.pvalues
    }).reset_index(drop=True)

    return {
        "model": model,
        "or_df": or_df,
        "auc": auc,
        "fpr" : fpr,
        "tpr": tpr,
        "thresholds": thresholds, 
        "y_test" : y_test, 
        "y_pred" : y_pred
    }

In [55]:
results = run_multivariable_lr(analysis_df, col_dict=cfg['univariable_lr'])
or_df = results['or_df']

In [56]:
def plot_roc(reults):
    fpr = results['fpr']
    tpr = results['tpr']
    auc = results['auc']

    fig, ax = plt.subplots(figsize=(6, 6))

    ax.plot(fpr, tpr, linewidth=2, label=f"AUC = {auc:.2f}", color = 'slateblue')
    ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1, label ='Random', color="cornflowerblue")
    
    ax.set_xlabel("False Positive Rate", labelpad = 10, fontsize = 16)
    ax.set_ylabel("True Positive Rate", labelpad = 10, fontsize = 16)
    ax.set_title("ROC curve", fontsize = 18)
    ax.legend(frameon=False, loc="lower right",  fontsize = 12)
    
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    
    plt.tight_layout()
    plt.show()

In [57]:
plot_roc(results)

In [58]:
def filter_or_plot_df(or_df, max_ci_upper=20):
    """Filter OR results to finite and visually stable estimates.

    Parameters
    ----------
    results_df : pandas.DataFrame
        Logistic regression results containing OR and confidence interval columns.
    max_ci_upper : float, default=20
        Remove rows with upper confidence interval above this value to avoid
        unreadable plots from unstable estimates.

    Returns
    -------
    pandas.DataFrame
        Filtered plotting dataframe.
    """
    plot_df = or_df.copy()

    plot_df = plot_df[
        np.isfinite(plot_df["OR"]) &
        np.isfinite(plot_df["ci_lower"]) &
        np.isfinite(plot_df["ci_upper"])
    ]

    plot_df = plot_df[
        (plot_df["ci_lower"] > 0) &
        (plot_df["ci_upper"] < max_ci_upper)
    ].copy()

    print(f"Removed {len(or_df) - len(plot_df)}")

    return plot_df


In [59]:
import re

def split_term(term):
    if term == "const":
        return "Intercept", ""
    
    if "__" in term:
        predictor, category = term.split('__', 1)
        return predictor, category
    
    return term, ""
    
or_df[["predictor", "category"]] = or_df["term"].apply(
    lambda x: pd.Series(split_term(x))
)

rename_dict = {
    "organism_bug": "Organism",
    "site": "Sample site",
    "tfc_group": "Surgery type",
    "prophylaxis_group": "Prophylaxis",
    "ethnicity_desc": "Ethnicity",
    "imd_decile": "Deprivation",
    "past_abx": "Prior antibiotics"
}

or_df["predictor"] = or_df["predictor"].replace(rename_dict)

In [60]:
or_df['is_categorical'] = or_df['category'] != ""

In [61]:
significant = or_df[or_df['p_value'] <0.05]
significant.sort_values(by = 'p_value', ascending = True)

In [62]:
or_df

In [16]:
def plot_variable_or(
    cat_results_df,
    predictor,
    title=None,
    label_col="category",
    figsize=(8, 5),
    max_ci_upper=20,
    annotate_significant=True,
):
    """Plot univariable odds ratios for one categorical predictor.

    Parameters
    ----------
    cat_results_df : pandas.DataFrame
        Categorical univariable LR results.
    predictor : str
        Predictor to plot.
    title : str, optional
        Plot title. If None, a default title is used.
    label_col : str, default="category"
        Column used for y-axis labels.
    figsize : tuple, default=(8, 5)
        Figure size.
    max_ci_upper : float, default=20
        Remove very wide confidence intervals from the plotted dataframe.
    annotate_significant : bool, default=True
        AddORtext labels for rows with p < 0.05.

    Returns
    -------
    matplotlib.axes.Axes
        The plot axis.
    """
    plot_df = filter_or_plot_df(cat_results_df, max_ci_upper=max_ci_upper)

    df_var = plot_df[plot_df["predictor"] == predictor].copy()
    df_var = df_var.sort_values("OR")
    df_var["label"] = df_var[label_col]
    df_var["significant"] = df_var["p_value"] < 0.05

    y_pos = np.arange(len(df_var))
    fig, ax = plt.subplots(figsize=figsize)

    df_nonsig = df_var[~df_var["significant"]]
    y_nonsig = y_pos[~df_var["significant"].values]

    ax.errorbar(
        df_nonsig["OR"],
        y_nonsig,
        xerr=[
            df_nonsig["OR"] - df_nonsig["ci_lower"],
            df_nonsig["ci_upper"] - df_nonsig["OR"],
        ],
        fmt="o",
        color="gray",
        ecolor="gray",
        elinewidth=1.5,
        capsize=3,
        markersize=6,
        alpha=0.8,
        label="p ≥ 0.05",
    )

    df_sig = df_var[df_var["significant"]]
    y_sig = y_pos[df_var["significant"].values]

    ax.errorbar(
        df_sig["OR"],
        y_sig,
        xerr=[
            df_sig["OR"] - df_sig["ci_lower"],
            df_sig["ci_upper"] - df_sig["OR"],
        ],
        fmt="o",
        color="navy",
        ecolor="navy",
        elinewidth=1.5,
        capsize=3,
        markersize=6,
        label="p < 0.05",
    )

    if annotate_significant:
        for x, y, or_val in zip(df_sig["OR"], y_sig, df_sig["OR"]):
            ax.text(
                x,
                y + 0.15,
                f"OR {or_val:.2f}",
                ha="center",
                va="bottom",
                fontsize=9,
                color="navy",
            )

    ax.axvline(1, color="black", linestyle="--", linewidth=1)
    ax.set_xscale("log")
    ax.set_xticks([0.25, 0.5, 1, 2, 4])
    ax.get_xaxis().set_major_formatter(mticker.ScalarFormatter())
    ax.ticklabel_format(style="plain", axis="x")

    ax.set_yticks(y_pos)
    ax.set_yticklabels(df_var["label"])
    ax.set_xlabel("Odds ratio (log scale)")

    if title is None:
        title = f"Association between {predictor} and ESBL infection"
    ax.set_title(title, pad=12)

    ax.grid(axis="x", linestyle=":", alpha=0.4)
    ax.grid(axis="y", visible=False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(frameon=False)

    plt.tight_layout()
    plt.show()

    return ax


In [17]:
plot_variable_or(or_df[or_df['is_categorical'] == True], predictor = 'Organism')

# Plot ORs -- FIX THIS FUNCTION

In [18]:
or_df['predictor'].unique()

In [19]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker


def get_forest_plot(or_df, max_ci_upper = 20):

    
    # -----------------------------
    # 1. Copy results
    # -----------------------------
    plot_df = or_df.copy()

    # remove intercept 
    plot_df = plot_df[plot_df['predictor'] != "Intercept"].copy()

    # filter out unstable
    plot_df = plot_df[
        np.isfinite(plot_df["OR"]) &
        np.isfinite(plot_df["ci_lower"]) &
        np.isfinite(plot_df["ci_upper"]) &
        (plot_df["ci_lower"] > 0) &
        (plot_df["ci_upper"] < max_ci_upper)
    ].copy()
    
    # -----------------------------
    # 3. Clean labels
    # -----------------------------
    category_map = {
    
            # Ethnicity
        "asian": "Asian",
        "black": "Black",
        "white": "White",
    
        # organisms
        "escherichia_coli": "E. coli",
        "klebsiella_pneumoniae": "K. pneumoniae",
        "klebsiella_oxytoca": "K. oxytoca",
        "proteus_mirabilis": "P. mirabilis",
    
        # sites
        "blood": "Blood",
        "drain": "Drain",
        "wound": "Wound",
        "tips_devices": "Catheter/device",
        "sputum": "Sputum",
        "low_vaginal": "Low vaginal",
        "high_vaginal": "High vaginal",
        "urine": "Urine",
        "tissue/biopsy": "Tissue / biopsy",
     
        # Surgical specialty
        "gynaecological_oncology": "Gynaecological oncology",
        "vascular": "Vascular surgery",
        "uro_nephro": "Urology / nephrology",
        "abdominal_gi": "Abdominal / GI surgery",
        "general_surgery": "General surgery",
        "ortho_plastics": "Orthopaedics / plastics",
        "Gynaecology": "Gynaecology",
        "cardiothoracic": "Cardiothoracic",
        "neuro_ent": "Neurosurgery / ENT",
        "obstetrics": "Obstetrics",
        "gynaecology" : "Gynaecology",
    
    
        # Prophylaxis
        "cephalosporin_aminoglycoside": "Cephalosporin&aminoglycoside",
        "cefuroxime": "Cefuroxime",
        "metronidazole": "Metronidazole",
        "glycopeptide_based": "Glycopeptide-based",
        "no_prophylaxis": "No prophylaxis",
        "aminoglycoside_based": "Aminoglycoside-based",
        "co-amoxiclav_(contains_penicillin)": "Co-amoxiclav",
        "clindamycin": "Clindamycin",
    
    }
    
    plot_df["categorical_clean"] = plot_df["category"].replace(category_map)

    plot_df['label'] = np.where(
        plot_df['is_categorical'],
        plot_df['predictor'] + ": " + plot_df['categorical_clean'],
        plot_df['predictor']
    )

    binary_continuous_order = [
        'age_at_admission', 'gender', 'asthma', 'copd', 'hypertension', 'type2_diabetes',
       'crp', 'temp', 'days_from_admission_to_first_culture', 
       'hospital_exposure_90d', 'colonisation',  'Prior antibiotics',
       'surgery_length']
    
    # -----------------------------
    # 4. Sort groups internally by OR
    # -----------------------------
    categorical_order = ['Organism', 'Sample site', 'Surgery type', 'Prophylaxis',
           'Ethnicity']
    
    plot_df["group"] = np.where(
        plot_df['is_categorical'],
        plot_df['predictor'], 
        'Clinical covariates'
    )

    group_order = ['Clinical covariates'] + categorical_order

    plot_df['group'] = pd.Categorical(
        plot_df['group'], 
        categories = group_order, 
        ordered= True
    )

    # sort within group
    
    plot_df = plot_df.sort_values(["group", "OR"], ascending=[True, False]).reset_index(drop=True)
    
    # -----------------------------
    # 5. Create y positions with gaps between groups
    # -----------------------------
    y_positions = []
    current_y = 0
    
    for grp in group_order:
    
        df_grp = plot_df[plot_df["group"] == grp]

        if len(df_grp) == 0:
            continue
    
        for _ in range(len(df_grp)):
            y_positions.append(current_y)
            current_y += 1
    
        current_y += 1  # blank line between groups
    
    plot_df["y"] = y_positions
    plot_df["significant"] = plot_df["p_value"] < 0.05
    
    # -----------------------------
    # 6. Plot
    # -----------------------------
    fig, ax = plt.subplots(figsize=(10, max(14, 0.45 * len(plot_df))))
    
    # non-significant
    df_nonsig = plot_df[~plot_df["significant"]]
    
    ax.errorbar(
        df_nonsig["OR"],
        df_nonsig["y"],
        xerr=[
            df_nonsig["OR"] - df_nonsig["ci_lower"],
            df_nonsig["ci_upper"] - df_nonsig["OR"]
        ],
        fmt="o",
        color="0.65",
        ecolor="0.65",
        elinewidth=2,      # thicker CI lines
        capsize=3,           # bigger caps
        capthick=1,
        markersize=8,       # bigger circles
        alpha=0.9,
    )
    
    # significant
    df_sig = plot_df[plot_df["significant"]]
    
    ax.errorbar(
        df_sig["OR"],
        df_sig["y"],
        xerr=[
            df_sig["OR"] - df_sig["ci_lower"],
            df_sig["ci_upper"] - df_sig["OR"]
        ],
        fmt="o",
        color="navy",
        ecolor="navy",
        elinewidth=2,
        capsize=3,
        capthick=1,
        markersize=8,
    )
    # annotate OR values
    for _, row in df_sig.iterrows():
        ax.text(
            row["OR"],
            row["y"] - 0.25,
            f"{row['OR']:.2f}",
            ha="center",
            va="bottom",
            fontsize=12,
            color="navy"
        )
    
    # reference line
    ax.axvline(1, color="black", linestyle="--", linewidth=1.2)
    
    # axes
    
    
    # axes
    ax.set_yticks(plot_df["y"])
    ax.set_yticklabels(plot_df["label"], fontsize=16)   # bigger category labels
    
    ax.set_xscale("log")
    ax.set_xticks([0.25, 0.5, 1, 2, 4])
    ax.get_xaxis().set_major_formatter(mticker.ScalarFormatter())
    
    ax.tick_params(axis='x', labelsize=15)  # bigger tick numbers
    
    ax.set_xlabel(
        "Odds ratio (95% CI, log scale)",
        fontsize=18,
        labelpad=10
    )
    
    ax.set_title(
        "Factors associated with ESBL infection",
        fontsize=22,
        weight="bold",
        pad=20
    )
    
    # style
    ax.grid(axis="x", linestyle=":", alpha=0.35, linewidth=1.2)
    ax.grid(axis="y", visible=False)
    
    # thicker reference line
    ax.axvline(1, color="black", linestyle="--", linewidth=2.5)
    
    # thicker axis lines
    ax.spines["bottom"].set_linewidth(1.5)
    ax.spines["left"].set_linewidth(1.5)
    
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    
    ax.invert_yaxis()
    
    plt.tight_layout()
    plt.savefig("figure_poster.png", bbox_inches='tight',  pad_inches=0.1)
    plt.show()

In [20]:
get_forest_plot(or_df)

# Stratify by TFC and infection site 

In [46]:
def fit_stratified_lr(
    df,
    strat_col,
    strat_value,
    outcome_col="esbl_status",
    categorical_cols=None,
    reference_categories=None,
    test_size=0.2,
    random_state=42,
    min_events=10
):
    import numpy as np
    import pandas as pd
    import statsmodels.api as sm
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import roc_auc_score

    df = df.copy()

    if categorical_cols is None:
        categorical_cols = ["ethnicity_desc", "organism_bug", "prophylaxis_group", "site", "tfc"]

    if reference_categories is None:
        reference_categories = {
            "organism_bug": "escherichia coli",
            "site": "blood",
            "ethnicity_desc": "white",
            "prophylaxis_group": "cefuroxime + metronidazole",
            "tfc": "general_surgery"
        }

    df_stratum = df[df[strat_col] == strat_value].copy()

    y = df_stratum[outcome_col]
    if y.dtype == "object":
        y = y.map({"ESBL": 1, "non-ESBL": 0})

    X = df_stratum.drop(columns=[outcome_col, strat_col], errors="ignore")

    cat_cols = [c for c in categorical_cols if c in X.columns and c != strat_col]

    X = pd.get_dummies(
        X,
        columns=cat_cols,
        drop_first=False,
        prefix_sep="__",
        dtype=int
    )

    for col, ref in reference_categories.items():
        if col != strat_col:
            X = X.drop(columns=f"{col}__{ref}", errors="ignore")

    X = X.apply(pd.to_numeric, errors="coerce")

    model_df = X.copy()
    model_df[outcome_col] = y.values
    model_df = model_df.dropna()

    y = model_df[outcome_col]
    X = model_df.drop(columns=[outcome_col])

    X = X.loc[:, X.nunique() > 1]

    n_events = int(y.sum())
    n_nonevents = int((y == 0).sum())

    if n_events < min_events or n_nonevents < min_events:
        raise ValueError(
            f"Too few events/non-events in {strat_col}={strat_value}. "
            f"Events={n_events}, non-events={n_nonevents}."
        )

    X = sm.add_constant(X, has_constant="add")

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
        stratify=y
    )

    print("Shape:", X_train.shape)
    print("Events:", y_train.sum())
    print("non-events:", (y_train ==0).sum())

    constant_cols = [
        c for c in X_train.columns[X_train.nunique() <= 1]]
    print('constant cols', constant_cols)

    dummy_cols = [c for c in X_train.columns if '__' in c]
    rare_cols = [c for c in dummy_cols if X_train[c].sum() <5]
    print('rare dummy cols:', rare_cols) 


    # remove rare columns

    constant_cols = [
        c for c in X_train.columns
        if c != 'const' and X_train[c].nunique() <=1]
    rare_cols = [
        c for c in X_train.columns
        if "__" in c and X_train[c].sum() <5
    ]
    cols_to_drop = list(set(constant_cols + rare_cols))

    X_train = X_train.drop(columns = cols_to_drop, errors='ignore')
    X_test = X_test.drop(columns = cols_to_drop, errors = 'ignore')

    model = sm.Logit(y_train, X_train).fit(disp=False)

    y_pred = model.predict(X_test)
    auc = roc_auc_score(y_test, y_pred)

    conf = model.conf_int()

    or_df = pd.DataFrame({
        "term": model.params.index,
        "OR": np.exp(model.params),
        "CI_lower": np.exp(conf[0]),
        "CI_upper": np.exp(conf[1]),
        "p_value": model.pvalues
    }).reset_index(drop=True)

    return {
        "strat_col": strat_col,
        "strat_value": strat_value,
        "n": len(model_df),
        "events": n_events,
        "non_events": n_nonevents,
        "auc_test": auc,
        "model": model,
        "or_df": or_df, 
        "X_test": X_test,
        "y_test": y_test,
        "y_pred_prob": y_pred
    }

# Stratify by TFC

In [67]:
def fit_lr_by_tfc(df, tfc_value, **kwargs):
    return fit_stratified_lr(
        df=df,
        strat_col="tfc",
        strat_value=tfc_value,
        **kwargs
    )

In [76]:
sub_analysis_df = analysis_df.drop(columns = [
    'prophylaxis_group', 'asthma', 'copd', 'hypertension',
    'type2_diabetes', 'age_at_admission',
    'crp', 'temp'], errors = 'ignore')
sub_analysis_df = sub_analysis_df[sub_analysis_df['organism_bug'] != 'klebsiella oxytoca']

In [77]:
sub_analysis_df.columns

In [71]:
result_tfc = fit_lr_by_tfc(sub_analysis_df, tfc_value='general_surgery')
print(f'n:{result_tfc["n"]}, events: {result_tfc["events"]}, auc_test: {result_tfc["auc_test"]}')
result_tfc = result_tfc["or_df"].sort_values("p_value")
print(result_tfc[result_tfc['p_value'] < 0.05])

In [72]:
result_tfc = fit_lr_by_tfc(sub_analysis_df, tfc_value='abdominal_gi')
print(f'n:{result_tfc["n"]}, events: {result_tfc["events"]}, auc_test: {result_tfc["auc_test"]}')
result_tfc = result_tfc["or_df"].sort_values("p_value")
print(result_tfc[result_tfc['p_value'] < 0.05])

In [73]:
result_tfc = fit_lr_by_tfc(sub_analysis_df, tfc_value='gynaecological_oncology')
print(f'n:{result_tfc["n"]}, events: {result_tfc["events"]}, auc_test: {result_tfc["auc_test"]}')
result_tfc = result_tfc["or_df"].sort_values("p_value")
print(result_tfc[result_tfc['p_value'] < 0.05])

In [74]:
result_tfc = fit_lr_by_tfc(sub_analysis_df, tfc_value='cardiothoracic')
print(f'n:{result_tfc["n"]}, events: {result_tfc["events"]}, auc_test: {result_tfc["auc_test"]}')
result_tfc = result_tfc["or_df"].sort_values("p_value")
print(result_tfc[result_tfc['p_value'] < 0.05])

In [75]:
result_tfc = fit_lr_by_tfc(sub_analysis_df, tfc_value='uro_nephro')
print(f'n:{result_tfc["n"]}, events: {result_tfc["events"]}, auc_test: {result_tfc["auc_test"]}')
result_tfc = result_tfc["or_df"].sort_values("p_value")
print(result_tfc[result_tfc['p_value'] < 0.05])

# Stratify by site

In [56]:
sub_analysis_df = sub_analysis_df.drop(columns = 'tfc')

In [57]:
def fit_lr_by_site(df, site_value, **kwargs):
    return fit_stratified_lr(
        df=df,
        strat_col="site",
        strat_value=site_value,
        **kwargs
    )

In [58]:
analysis_df['site'].unique()

In [59]:
result_site = fit_lr_by_site(sub_analysis_df, site_value='sputum')
print(result_site["n"], result_site["events"], result_site["auc_test"])
result_site_df = result_site["or_df"].sort_values("p_value")
result_site_df[result_site_df['p_value'] < 0.05]

In [60]:
result_site = fit_lr_by_site(sub_analysis_df, site_value='urine')
print(result_site["n"], result_site["events"], result_site["auc_test"])
result_site_df = result_site["or_df"].sort_values("p_value")
result_site_df[result_site_df['p_value'] < 0.05]

In [61]:
result_site = fit_lr_by_site(sub_analysis_df, site_value='wound')
print(result_site["n"], result_site["events"], result_site["auc_test"])
result_site_df = result_site["or_df"].sort_values("p_value")
result_site_df[result_site_df['p_value'] < 0.05]

# Sensitivity analysis 

In [63]:
def load_analysis_data(path="../data/interim/analysis_df.csv", surgery_only=True):
    """Load the analysis dataset and optionally restrict to surgical patients.

    Parameters
    ----------
    path : str or pathlib.Path
        Path to the analysis dataframe CSV.
    surgery_only : bool, default=True
        If True, keep only rows where `surgery_before_infection == 1`.

    Returns
    -------
    pandas.DataFrame
        Loaded and optionally filtered analysis dataframe.
    """
    analysis_df = pd.read_csv(path)

    if surgery_only:
        analysis_df = analysis_df[analysis_df["surgery_before_infection"] == 1].copy()

    return analysis_df
    
sensitivity_df = load_analysis_data()
sensitivity_df.columns

In [64]:
episodes_per_patient = sensitivity_df.groupby('subject')['infection_id'].agg('nunique').reset_index()

multiple_eps = episodes_per_patient[episodes_per_patient['infection_id'] > 1]

infection_counts = multiple_eps['infection_id'].value_counts()
ax = sns.barplot(infection_counts)
ax.set(ylabel = 'Count', xlabel= 'Episodes')

In [65]:
=

In [66]:
first_episode_df = (
    sensitivity_df
    .sort_values(["subject", "infection_id"])
    .drop_duplicates("subject", keep="first")
)

In [67]:
episode_counts = sensitivity_df["subject"].value_counts()

episode_counts.describe()

In [68]:
n_patients = sensitivity_df["subject"].nunique()
n_episodes = len(sensitivity_df)
n_repeated_patients = (episode_counts > 1).sum()


print("Episodes:", n_episodes)
print("Unique patients:", n_patients)
print("Patients with >1 episode:", n_repeated_patients)



In [46]:
first_ep_df = pd.read_csv('first_ep_data.csv')
sensitivity_result = run_multivariable_lr(first_ep_df, col_dict=cfg['univariable_lr'])

In [47]:
model = results["model"]
y_test = results["y_test"]
y_pred_prob = results["y_pred_prob"]

In [ ]:
comparison = results["or_df"].merge(
    sensitivity_result["or_df"],
    on="term",
    suffixes=("_main", "_first_episode")
)

comparison[[
    "term",
    "OR_main",
    "ci_lower_main",
    "ci_upper_main",
    "p_value_main",
    "OR_first_episode",
    "ci_lower_first_episode",
    "ci_upper_first_episode",
    "p_value_first_episode"
]]

In [ ]:
print("Main model AUC:", results["auc"])
print("First-episode sensitivity AUC:", sensitivity_result["auc"])

# GEE regression 

In [ ]:
def run_gee_logistic(
    df,
    outcome_col="esbl_status",
    cluster_col="subject",
    categorical_cols=None,
    reference_categories=None
):
    import numpy as np
    import pandas as pd
    import statsmodels.api as sm

    if categorical_cols is None:
        categorical_cols = [
            "ethnicity_desc",
            "organism_bug",
            "prophylaxis_group",
            "site",
            "tfc"
        ]

    if reference_categories is None:
        reference_categories = {
            "organism_bug": "escherichia coli",
            "site": "blood",
            "ethnicity_desc": "white",
            "prophylaxis_group": "cefuroxime + metronidazole",
            "tfc": "general_surgery"
        }

    df = df.copy()

    y = df[outcome_col]
    if y.dtype == "object":
        y = y.map({"ESBL": 1, "non-ESBL": 0})

    X = df.drop(columns=[outcome_col, cluster_col], errors="ignore")

    X = pd.get_dummies(
        X,
        columns=[c for c in categorical_cols if c in X.columns],
        prefix_sep="__",
        drop_first=False,
        dtype=int
    )

    for col, ref in reference_categories.items():
        X = X.drop(columns=f"{col}__{ref}", errors="ignore")

    X = X.apply(pd.to_numeric, errors="coerce")

    model_df = X.copy()
    model_df[outcome_col] = y.values
    model_df[cluster_col] = df[cluster_col].values
    model_df = model_df.dropna()

    y = model_df[outcome_col]
    groups = model_df[cluster_col]
    X = model_df.drop(columns=[outcome_col, cluster_col])

    # Drop zero-variance columns
    X = X.loc[:, X.nunique() > 1]

    X = sm.add_constant(X, has_constant="add")

    gee_model = sm.GEE(
        y,
        X,
        groups=groups,
        family=sm.families.Binomial(),
        cov_struct=sm.cov_struct.Exchangeable()
    )

    gee_result = gee_model.fit()

    conf = gee_result.conf_int()

    or_df = pd.DataFrame({
        "term": gee_result.params.index,
        "OR": np.exp(gee_result.params),
        "ci_lower": np.exp(conf[0]),
        "ci_upper": np.exp(conf[1]),
        "p_value": gee_result.pvalues
    }).reset_index(drop=True)

    return {
        "model": gee_result,
        "or_df": or_df,
        "n": len(model_df),
        "n_clusters": model_df[cluster_col].nunique(),
        "events": int(y.sum())
    }

In [ ]:
gee_result = run_gee_logistic(sensitivity_df)

gee_result["or_df"].sort_values("p_value").head(20)

In [ ]:
lr_result = run_multivariable_lr(analysis_df)

comparison = lr_result["or_df"].merge(
    gee_result["or_df"],
    on="term",
    suffixes=("_LR", "_GEE")
)

comparison[[
    "term",
    "OR_LR", "ci_lower_LR", "ci_upper_LR", "p_value_LR",
    "OR_GEE", "ci_lower_GEE", "ci_upper_GEE", "p_value_GEE"
]]

# What to report 

In [ ]:
unique_patients = raw_df["subject"].nunique()
unique_episodes = raw_df['infection_id'].nunique()
multiple_infections = raw_df.groupby('subject')['infection_id'].nunique().reset_index()
has_multiple_infections = len(multiple_infections[multiple_infections['infection_id'] >1])

In [ ]:
print(
    f"There are {unique_episodes} unique infection episodes from {unique_patients} unique patients. {has_multiple_infections} patients ({round(has_multiple_infections/unique_patients,2)}%) have more than one infection episode.")
print(raw_df["esbl_status"].value_counts())
raw_df["esbl_status"].value_counts(normalize=True)

In [ ]:
missing = analysis_df.isna().sum().sort_values(ascending=False)
missing_pct = analysis_df.isna().mean().sort_values(ascending=False) * 100

In [ ]:
# Model diagnostics

model.mle_retvals

from statsmodels.stats.outliers_influence import variance_inflation_factor


In [ ]:
# Confusion matrix

from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import classification_report

# Convert probabilities to binary predictions
threshold = 0.5

y_pred_class = (y_pred_prob >= threshold).astype(int)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred_class)

print(cm)

# Plot
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()

# Classification metrics
print(classification_report(y_test, y_pred_class))

In [ ]:
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt

prob_true, prob_pred = calibration_curve(
    y_test,
    y_pred_prob,
    n_bins=10
)

plt.figure(figsize=(6,6))

# Perfect calibration line
plt.plot([0,1], [0,1], linestyle='--')

# Model calibration
plt.plot(prob_pred, prob_true, marker='o')

plt.xlabel("Predicted probability")
plt.ylabel("Observed proportion ESBL")
plt.title("Calibration Curve")

plt.show()